In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.ndimage import median_filter

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [19]:
train.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
0,-0.301962,-0.674429,-0.813442,-0.986180,-1.055257,-1.043944,-1.221650,-1.047900,-1.058251,-0.767975,...,0.912867,0.705133,0.312869,0.216079,0.066146,-0.128556,-0.528392,-0.601232,-0.874030,-0.920702
1,-0.679885,-0.839446,-0.939951,-1.007480,-0.884820,-0.791173,-0.266799,-0.268019,0.127274,0.563845,...,-1.190800,-0.994198,-0.933952,-0.826262,-0.409902,-0.259148,0.276285,0.507796,0.596902,0.977113
2,-0.123810,0.366783,0.754148,1.005530,0.930500,0.536824,0.335199,-0.295252,-0.775824,-1.073319,...,0.627682,0.333898,-0.248565,-0.721124,-0.869798,-1.158567,-0.696588,-0.459882,0.131853,0.544116
3,-0.910338,-0.901496,-0.811174,-0.723511,-0.272930,0.175472,0.743182,0.739125,1.045961,0.846212,...,0.212179,-0.307177,-0.652537,-0.946943,-6.319450,-0.797647,-0.530532,-0.276249,0.371958,0.732336
4,-0.575937,-0.203204,0.217311,0.512397,0.819564,1.047765,0.847104,0.611637,NaN,-0.124647,...,-0.690263,-0.466631,-0.066552,0.321127,0.698367,0.932146,1.039476,0.957863,0.612729,0.360576


In [20]:
test.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
0,1.133991,0.974741,0.778325,0.487721,0.249792,-0.398965,-0.531340,-0.682951,-1.112844,-1.105730,...,-0.991552,-1.070664,-1.015583,-0.966808,-0.820162,-0.484321,-0.026678,0.285303,0.480518,0.705384
1,0.859651,0.890513,0.591865,0.349314,-0.141356,-0.345916,-0.682196,-0.835734,-0.813710,-0.782985,...,0.924108,1.002806,0.667275,0.297175,-0.121421,-0.220042,-0.519712,-0.788980,-0.855378,-0.917613
2,1.057161,1.000076,0.895016,0.576635,0.476865,-0.056215,-0.251354,-0.375827,-0.701685,-0.957806,...,0.692908,0.368526,0.207372,-0.079378,-0.341062,-0.648833,-0.960609,-1.113595,-1.016396,-1.021047
3,0.512833,0.060535,-0.390935,-0.967501,-1.068736,-0.913047,-0.681511,-0.523403,-0.126449,0.334413,...,0.094271,-0.180347,-0.667616,-0.951487,-0.834389,-0.878719,-0.625129,-0.255535,0.272052,0.583315
4,-0.334186,-0.130046,0.315546,0.780386,0.806072,0.854475,0.759626,0.522307,0.356519,0.113692,...,0.694435,0.868146,0.833548,0.856095,0.502581,0.425936,-0.190167,-0.303861,-0.509646,-0.730039


In [21]:
def preprocess(frame):
    x = frame.to_numpy(dtype=np.float64, copy=True)
    positions = np.arange(x.shape[1])
    for row in x:
        valid = np.isfinite(row)
        if valid.any():
            row[~valid] = np.interp(positions[~valid], positions[valid], row[valid])
        else:
            row[:] = 0.0

    local_median = median_filter(x, size=(1, 3), mode='nearest')
    local_error = x - local_median
    spike_scale = 1.4826 * np.median(
        np.abs(local_error - np.median(local_error, axis=1, keepdims=True)), axis=1
    ) + 1e-6
    spike_mask = np.abs(local_error) > np.maximum(8.0 * spike_scale[:, None], 2.0)
    x[spike_mask] = local_median[spike_mask]
    return x

def reconstruction_features(x):
    n_rows, length = x.shape
    time = np.arange(length)

    frequency_grid = np.linspace(3.5, 10.5, 141)
    fourier_dictionary = np.exp(
        -2j * np.pi * np.outer(frequency_grid, time) / length
    )
    centered = x - np.median(x, axis=1, keepdims=True)
    initial_frequency = frequency_grid[
        np.abs(centered @ fourier_dictionary.T).argmax(axis=1)
    ]

    fitted = np.empty_like(x)
    selected_frequency = np.empty(n_rows)
    for row_index, (row, seed_frequency) in enumerate(zip(x, initial_frequency)):
        best_loss = np.inf
        for frequency in np.linspace(seed_frequency - 0.075, seed_frequency + 0.075, 7):
            design = np.column_stack([
                np.ones(length),
                np.sin(2 * np.pi * frequency * time / length),
                np.cos(2 * np.pi * frequency * time / length),
                np.sin(4 * np.pi * frequency * time / length),
                np.cos(4 * np.pi * frequency * time / length),
            ])
            keep = np.ones(length, dtype=bool)
            for _ in range(3):
                coefficients = np.linalg.lstsq(design[keep], row[keep], rcond=None)[0]
                residual = row - design @ coefficients
                keep = np.abs(residual) <= np.quantile(np.abs(residual), 0.75)
            loss = np.median(np.abs(residual))
            if loss < best_loss:
                best_loss = loss
                fitted[row_index] = design @ coefficients
                selected_frequency[row_index] = frequency

    residual = x - fitted
    residual_scale = 1.4826 * np.median(
        np.abs(residual - np.median(residual, axis=1, keepdims=True)), axis=1
    ) + 1e-4
    absolute_residual = np.abs(residual) / residual_scale[:, None]

    features = {
        'residual_q85': np.quantile(absolute_residual, 0.85, axis=1),
        'residual_q90': np.quantile(absolute_residual, 0.90, axis=1),
        'residual_q95': np.quantile(absolute_residual, 0.95, axis=1),
        'residual_max': np.max(absolute_residual, axis=1),
        'residual_rms': np.sqrt(np.mean(residual ** 2, axis=1)) / residual_scale,
    }
    residual_windows = np.lib.stride_tricks.sliding_window_view(residual, 12, axis=1)
    features['local_bias'] = (
        np.max(np.abs(np.mean(residual_windows, axis=2)), axis=1) / residual_scale
    )
    features['local_error'] = (
        np.max(np.sqrt(np.mean(residual_windows ** 2, axis=2)), axis=1) / residual_scale
    )

    signal_scale = 1.4826 * np.median(
        np.abs(x - np.median(x, axis=1, keepdims=True)), axis=1
    ) + 1e-8
    signal_windows = np.lib.stride_tricks.sliding_window_view(x, 24, axis=1)
    features['level_shift'] = np.abs(np.mean(x, axis=1))
    features['flatness'] = np.min(np.std(signal_windows, axis=2), axis=1) / signal_scale
    return features

x_train = preprocess(train)
x_test = preprocess(test)
train_features = reconstruction_features(x_train)
test_features = reconstruction_features(x_test)

In [22]:
def upper_tail_probability(reference, values):
    ordered = np.sort(reference)
    count = len(ordered) - np.searchsorted(ordered, values, side='left')
    return (count + 0.5) / (len(ordered) + 1.0)

def lower_tail_probability(reference, values):
    ordered = np.sort(reference)
    count = np.searchsorted(ordered, values, side='right')
    return (count + 0.5) / (len(ordered) + 1.0)

reconstruction_names = [
    'residual_q85', 'residual_q90', 'residual_q95',
    'residual_rms', 'local_bias', 'local_error', 'level_shift'
]

evidences = [
    -np.log10(upper_tail_probability(train_features[name], test_features[name]))
    for name in reconstruction_names
]

evidences.append(
    -np.log10(lower_tail_probability(train_features['flatness'], test_features['flatness']))
)

reconstruction_evidence = np.mean(evidences, axis = 0)

def fractional_rank(values):
    order = np.argsort(values, kind='mergesort')
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = np.arange(len(values), dtype=float)
    return ranks / max(len(values) - 1, 1)

raw_score = fractional_rank(reconstruction_evidence)
scores = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())
scores = np.clip(scores, 0.0, 1.0)